# Q1: 数据使用非典型性对学术成功的影响

研究问题：数据使用非典型性（Atypicality）如何影响学术成功？

因变量：H_index, Productivity, C3, C5
核心自变量：Atypicality_of_datasets_original_1
控制变量：Average_Team_Size, Avg_Citation_Without_Self, Academic_Age, Topic_Diversity

模型：Negative Binomial / OLS / GLM

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

from src.data_loader import load_config
from src.regression import run_ols, run_nb, run_glm, save_results
from src.visualization import plot_forest_chart, save_fig
import pandas as pd
import statsmodels.api as sm

In [ ]:
config = load_config()
data_path = os.path.join('studies', 'Q1', 'q1_regression_ready.csv')
df = pd.read_csv(data_path)
print(f'数据维度: {df.shape}')
df.head()

## 1. Negative Binomial 回归（计数因变量：H_index, Productivity, C3, C5）

In [ ]:
x_var = 'Atypicality_of_datasets_original_1'
controls = ['Average_Team_Size', 'Avg_Citation_Without_Self', 'Academic_Age', 'Topic_Diversity']
controls = [c for c in controls if c in df.columns]

nb_results = {}
for y_var in ['H_index', 'Productivity', 'C3', 'C5']:
    if y_var in df.columns:
        nb_results[y_var] = run_nb(df, y_var, x_var, controls)
        if nb_results[y_var]['converged']:
            print(f'{y_var}: coef={nb_results[y_var]["coef"]:.4f}, p={nb_results[y_var]["pvalue"]:.4f}')

## 2. OLS 回归（连续因变量）

In [ ]:
ols_results = {}
for y_var in ['C3', 'C5']:
    if y_var in df.columns:
        ols_results[y_var] = run_ols(df, y_var, x_var, controls)
        if ols_results[y_var]['converged']:
            print(f'{y_var}: coef={ols_results[y_var]["coef"]:.4f}, p={ols_results[y_var]["pvalue"]:.4f}')

## 3. Forest Plot

In [ ]:
all_results = {**nb_results, **ols_results}
save_results(all_results, os.path.join('studies', 'Q1', 'results', 'q1_regression_results.csv'))
fig = plot_forest_chart(all_results, title='Q1: Effect of Data Atypicality on Academic Success')
save_fig(fig, os.path.join('studies', 'Q1', 'figures', 'q1_forest_plot.pdf'))
plt.show()

## 4. 鲁棒性检验：截尾趋势

In [ ]:
from src.visualization import plot_binned_scatter

if 'C3' in df.columns:
    fig = plot_binned_scatter(df, x_var, 'C3', title='Q1: Binned Scatter - Atypicality vs C3')
    save_fig(fig, os.path.join('studies', 'Q1', 'figures', 'q1_binned_scatter_c3.pdf'))
    plt.show()